In [92]:
import pandas as pd

customer_df = pd.read_csv("customer.csv")
cars_df = pd.read_csv('cars.csv')



print(customer_df.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        50 non-null     int64 
 1   gender     50 non-null     object
 2   review     50 non-null     object
 3   education  50 non-null     object
 4   purchased  50 non-null     object
dtypes: int64(1), object(4)
memory usage: 2.1+ KB
None


In [93]:
customer_df.head(5)

,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No


## Ordinal Encoder
- There are 4 Categorical columns gender, review, education & purchaged. out of which gender and purchaged are Nomial data whereas review & education are the Ordinal data.
- purchaged: will the customer purchage the recommanded product after their own purchaged items or not ?

In [94]:
customer_df['review'].value_counts()

review
Poor       18
Good       18
Average    14
Name: count, dtype: int64

In [95]:
customer_df['education'].value_counts()

education
PG        18
School    16
UG        16
Name: count, dtype: int64

- `OHE`: Suitable for Nominal data
- `OrdinalEncoder`: suitable for Ordinal data
- `LabelEncoder`: suitable for Target Cat. Column only.

In [96]:
cust_df = customer_df.iloc[:,2:]

cust_df

,review,education,purchased
0,Average,School,No
1,Poor,UG,No
2,Good,PG,No
3,Good,PG,No
4,Average,UG,No
5,Average,School,Yes
6,Good,School,No
7,Poor,School,Yes
8,Average,UG,No
9,Good,UG,Yes


#### The Golden Rule

- Anything that learns from data must be fit only on the training set.
- This is one of the most important concepts in preprocessing because it is directly related to data leakage.
- For any scaler, encoder, imputer, PCA, or transformer: use `fit_transform()` on `X_train` and only `transform()` on `X_test`, because the model should learn parameters only from the training data and never from the test data (to avoid data leakage).
- `fit()` + `transform()` and `fit_transform()` are functionally equivalent; `fit_transform()` is just a shorter way to perform both steps together.

In [97]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(cust_df.iloc[:,:2], cust_df.iloc[:, -1], test_size=0.2, random_state=42)

X_train.head(5)

,review,education
12,Poor,School
4,Average,UG
37,Average,PG
8,Average,UG
3,Good,PG


- Since review & education are the ordinal data, we try to encode these columns with OrdinalEncoder of sklearn.
- OrdinalEncoder encodes the ordinal data in its order as 0,1,2,3,...
- eg.,` ['Poor','Average','Good']` ---->>> `[0,1,2]`

In [98]:
X_test

,review,education
13,Average,School
39,Poor,PG
30,Average,UG
45,Poor,PG
17,Poor,UG
48,Good,UG
26,Poor,PG
25,Good,School
32,Average,UG
19,Poor,PG


In [99]:
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

oe = OrdinalEncoder(categories=[['Poor','Average','Good'], ['School','UG','PG']])


X_train = oe.fit_transform(X_train)
X_test = oe.transform(X_test)

- We can see the proof as: `X_test` has  `[Average, School]` in its first row and after encoding it becomes `[1,0]`

In [100]:
X_test

array([[1., 0.],
       [0., 2.],
       [1., 1.],
       [0., 2.],
       [0., 1.],
       [2., 1.],
       [0., 2.],
       [2., 0.],
       [1., 1.],
       [0., 2.]])

In [101]:
oe.categories_

[array(['Poor', 'Average', 'Good'], dtype=object),
 array(['School', 'UG', 'PG'], dtype=object)]

## Label Encoder
- Since `purchaged` is a Categorical Target Variable, we encoded it with LabelEncoder.

In [102]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

le.fit(Y_train)

Y_train = le.transform(Y_train)
Y_test = le.transform(Y_test)

In [103]:
le.classes_

array(['No', 'Yes'], dtype=object)

## One Hot Encoding
- we do this encoding on a nominal data.

In [104]:
cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   brand          8128 non-null   object
 1   km_driven      8128 non-null   int64 
 2   fuel           8128 non-null   object
 3   owner          8128 non-null   object
 4   selling_price  8128 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 317.6+ KB


- There are three Nominal columns brand, fuel & owner

In [105]:
cars_df.head(5)

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


- brand has many unique values while fuel & owner has few.
- Without column transfarmer its a little difficult to encode all three together.
- So, We will encode fuel & owner together and brand separatly and then merge them both at the end.

In [106]:
print(cars_df['brand'].value_counts())

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64


In [107]:
print(cars_df['fuel'].value_counts())
print(cars_df['owner'].value_counts())

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64
owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64


# One-Hot Encoding Using Pandas (`pd.get_dummies()`)

One-Hot Encoding converts categorical data into numerical columns so that Machine Learning algorithms can understand and use them.

---

# Why Do We Need One-Hot Encoding?

Machine Learning models cannot directly understand text categories.

Example:

| City |
|--------|
| Delhi |
| Mumbai |
| Pune |

If we leave them as strings, most ML algorithms will throw an error.

One-Hot Encoding converts them into binary (0/1) columns:

| City_Delhi | City_Mumbai | City_Pune |
|------------|-------------|-----------|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

Each category gets its own column.

---

# Syntax

```python
pd.get_dummies(
    data,
    prefix=None,
    prefix_sep='_',
    dummy_na=False,
    columns=None,
    sparse=False,
    drop_first=False,
    dtype=None
)
```

---

# Sample Dataset

```python
import pandas as pd

df = pd.DataFrame({
    'City': ['Delhi', 'Mumbai', 'Pune', 'Delhi'],
    'Gender': ['Male', 'Female', 'Male', 'Female']
})

print(df)
```

Output:

| City | Gender |
|--------|---------|
| Delhi | Male |
| Mumbai | Female |
| Pune | Male |
| Delhi | Female |

---

# 1. Basic One-Hot Encoding

```python
pd.get_dummies(df)
```

Output:

| City_Delhi | City_Mumbai | City_Pune | Gender_Female | Gender_Male |
|------------|-------------|-----------|---------------|-------------|
| 1 | 0 | 0 | 0 | 1 |
| 0 | 1 | 0 | 1 | 0 |
| 0 | 0 | 1 | 0 | 1 |
| 1 | 0 | 0 | 1 | 0 |

Pandas automatically detects object/string columns and encodes them.

---

# Important Parameters

---

# 1. data

The DataFrame or Series to encode.

```python
pd.get_dummies(df)
```

or

```python
pd.get_dummies(df['City'])
```

---

# 2. columns

Used when you want to encode only specific columns.

### Example

```python
pd.get_dummies(df, columns=['City'])
```

Output:

| Gender | City_Delhi | City_Mumbai | City_Pune |
|---------|------------|-------------|-----------|
| Male | 1 | 0 | 0 |
| Female | 0 | 1 | 0 |
| Male | 0 | 0 | 1 |
| Female | 1 | 0 | 0 |

Only the City column is encoded.

Gender remains unchanged.

---

# 3. drop_first

Removes the first category.

Used to avoid the Dummy Variable Trap (Multicollinearity). 
- Multicollinearity occurs when one feature can be predicted from one or more other features, causing redundant information in the dataset.
- basically, to avoid Dummy Variable Trap, we need to remove all dependant columns.(there should not be any relationship between columns).
- Initially: there is a relationship, sum of each rows = 1
- To avoid it we need to remove any column, generally first column.
### Example

```python
pd.get_dummies(df, columns=['City'], drop_first=True)
```

Output:

| Gender | City_Mumbai | City_Pune |
|---------|-------------|-----------|
| Male | 0 | 0 |
| Female | 1 | 0 |
| Male | 0 | 1 |
| Female | 0 | 0 |

Notice:

- Delhi column removed
- Only k−1 columns created

If all remaining columns are 0:

```text
City = Delhi
```

---

## Why Use drop_first?

Without dropping:

```text
Delhi + Mumbai + Pune = 1
```

One column can always be predicted from the others.

This creates multicollinearity in models like:

- Linear Regression
- Logistic Regression

Therefore:

```python
drop_first=True
```

is often recommended for linear models.

For tree-based models (Decision Tree, Random Forest, XGBoost), it is usually unnecessary.

---

# 4. dummy_na

Creates an extra column for missing values (NaN).

### Example

```python
df = pd.DataFrame({
    'City': ['Delhi', 'Mumbai', None, 'Pune']
})
```

Without:

```python
pd.get_dummies(df)
```

Output:

| City_Delhi | City_Mumbai | City_Pune |
|------------|-------------|-----------|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 0 |
| 0 | 0 | 1 |

The NaN row becomes all zeros.

---

With:

```python
pd.get_dummies(df, dummy_na=True)
```

Output:

| City_Delhi | City_Mumbai | City_Pune | City_nan |
|------------|-------------|-----------|----------|
| 1 | 0 | 0 | 0 |
| 0 | 1 | 0 | 0 |
| 0 | 0 | 0 | 1 |
| 0 | 0 | 1 | 0 |

Now missing values have their own column.

---

# 5. dtype

Specifies the datatype of generated columns.

### Example

```python
pd.get_dummies(df, dtype=int)
```

Output:

```text
0 and 1 stored as integers
```

Without specifying:

```python
pd.get_dummies(df)
```

Pandas may create:

```text
True / False
```

or

```text
uint8
```

depending on the version.

Common choice:

```python
dtype=int
```

---

# 6. prefix

Adds a custom prefix to generated column names.

### Example

```python
pd.get_dummies(
    df,
    columns=['City'],
    prefix='Location'
)
```

Output:

| Location_Delhi | Location_Mumbai | Location_Pune |
|----------------|-----------------|---------------|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

---

# 7. prefix_sep

Controls the separator between prefix and category.

Default:

```python
_
```

Example:

```python
pd.get_dummies(
    df,
    columns=['City'],
    prefix='Location',
    prefix_sep='-'
)
```

Output:

```text
Location-Delhi
Location-Mumbai
Location-Pune
```

---

# 8. sparse

Creates sparse columns to save memory.

A sparse column is a column that contains mostly zeros (or missing values) and very few non-zero values.

Instead of storing:

`0 0 0 0 0 0 0 1 0 0 0`

a sparse representation stores only:

`Position 8 = 1`

This saves a lot of memory and can speed up computations.

Useful when:

- Huge datasets
- Thousands of categories
- Mostly zeros

Example:

```python
pd.get_dummies(df, sparse=True)
```

Usually not required for small datasets.

---

# Complete Example

```python
import pandas as pd

df = pd.DataFrame({
    'City': ['Delhi', 'Mumbai', 'Pune', 'Delhi'],
    'Gender': ['Male', 'Female', 'Male', 'Female']
})

encoded_df = pd.get_dummies(
    df,
    columns=['City', 'Gender'],
    drop_first=True,
    dtype=int
)

print(encoded_df)
```

Output:

| City_Mumbai | City_Pune | Gender_Male |
|-------------|-----------|-------------|
| 0 | 0 | 1 |
| 1 | 0 | 0 |
| 0 | 1 | 1 |
| 0 | 0 | 0 |

---

# Interview Notes

### What is One-Hot Encoding?

Converts categorical variables into multiple binary (0/1) columns.

---

### Why not Label Encode nominal data?

Because Label Encoding introduces an artificial order.

Example:

```text
Delhi  -> 0
Mumbai -> 1
Pune   -> 2
```

The model may incorrectly assume:

```text
Pune > Mumbai > Delhi
```

One-Hot Encoding avoids this problem.

---

### What is the Dummy Variable Trap?

When all dummy columns are included, one column becomes perfectly predictable from the others.

Solution:

```python
drop_first=True
```

---

### When should One-Hot Encoding be used?

Use for:

- Nominal categorical data
- Categories with no natural order

Examples:

- City
- Country
- Gender
- Color

---

# Quick Summary

| Parameter | Purpose |
|------------|----------|
| data | DataFrame or Series to encode |
| columns | Specific columns to encode |
| drop_first | Remove first category (k−1 columns) |
| dummy_na | Create separate column for NaN |
| dtype | Data type of generated columns |
| prefix | Custom column prefix |
| prefix_sep | Separator between prefix and category |
| sparse | Memory-efficient sparse output |

---

# Golden Rule

Use One-Hot Encoding for Nominal Categories (no order), and consider `drop_first=True` when working with linear models to avoid multicollinearity.

In [108]:
cars_df

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000
...,...,...,...,...,...
8123,Hyundai,110000,Petrol,First Owner,320000
8124,Hyundai,119000,Diesel,Fourth & Above Owner,135000
8125,Maruti,120000,Diesel,First Owner,382000
8126,Tata,25000,Diesel,First Owner,290000


In [109]:
cars_df.value_counts('fuel')

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

In [110]:
cars_df.value_counts('owner')

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

In [111]:
# pd.get_dummies(cars_df, columns=['fuel', 'owner'], sparse=True, dtype=int).dtypes
pd.get_dummies(cars_df, columns=['fuel', 'owner'], sparse=True, dtype=int)


,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,0,1,0,0,1,0,0,0,0
1,Skoda,120000,370000,0,1,0,0,0,0,1,0,0
2,Honda,140000,158000,0,0,0,1,0,0,0,0,1
3,Hyundai,127000,225000,0,1,0,0,1,0,0,0,0
4,Maruti,120000,130000,0,0,0,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,0,1,1,0,0,0,0
8124,Hyundai,119000,135000,0,1,0,0,0,1,0,0,0
8125,Maruti,120000,382000,0,1,0,0,1,0,0,0,0
8126,Tata,25000,290000,0,1,0,0,1,0,0,0,0


## K-1 One Hot encoding

In [112]:
pd.get_dummies(cars_df, columns=['fuel', 'owner'], sparse=False, dtype=int, drop_first=True)


,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,1,0,0,0,0,0,0
1,Skoda,120000,370000,1,0,0,0,1,0,0
2,Honda,140000,158000,0,0,1,0,0,0,1
3,Hyundai,127000,225000,1,0,0,0,0,0,0
4,Maruti,120000,130000,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,1,0,0,0,0
8124,Hyundai,119000,135000,1,0,0,1,0,0,0
8125,Maruti,120000,382000,1,0,0,0,0,0,0
8126,Tata,25000,290000,1,0,0,0,0,0,0


## One Hot encoding using sklearn

In [113]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(cars_df.iloc[:, :4], cars_df.iloc[:, -1], test_size=0.2, random_state=42)
x_train

,brand,km_driven,fuel,owner
6518,Tata,2560,Petrol,First Owner
6144,Honda,80000,Petrol,Second Owner
6381,Hyundai,150000,Diesel,Fourth & Above Owner
438,Maruti,120000,Diesel,Second Owner
5939,Maruti,25000,Petrol,First Owner
...,...,...,...,...
5226,Mahindra,120000,Diesel,First Owner
5390,Maruti,80000,Diesel,Second Owner
860,Hyundai,35000,Petrol,First Owner
7603,Maruti,27000,Diesel,First Owner


In [114]:
from sklearn.preprocessing import OneHotEncoder

OHE = OneHotEncoder(drop = 'first', sparse_output=False)

x_train_new = OHE.fit_transform(x_train[['fuel','owner']])
x_test_new = OHE.transform(x_test[['fuel','owner']])

x_train_new

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.]], shape=(6502, 7))

In [115]:
np.hstack((x_train[['brand', 'km_driven']].values,x_train_new))

array([['Tata', 2560, 0.0, ..., 0.0, 0.0, 0.0],
       ['Honda', 80000, 0.0, ..., 1.0, 0.0, 0.0],
       ['Hyundai', 150000, 1.0, ..., 0.0, 0.0, 0.0],
       ...,
       ['Hyundai', 35000, 0.0, ..., 0.0, 0.0, 0.0],
       ['Maruti', 27000, 1.0, ..., 0.0, 0.0, 0.0],
       ['Maruti', 70000, 0.0, ..., 1.0, 0.0, 0.0]],
      shape=(6502, 9), dtype=object)

## One Hot encoding using top Categories

Mental model

`df['col']`

→ One-dimensional object (Series)

→ Count values.

`df[['col']]`

→ Two-dimensional object (DataFrame)

→ Count duplicate rows.

`cars_df[['brand', 'km_driven']].value_counts()`

→ counts unique combinations of brand and model.

In [116]:
cars_df['brand'].value_counts()

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

In [ ]:
count = cars_df[['brand']].value_counts()
thrashold = 100


In [118]:
type(count)

pandas.core.series.Series

In [119]:
repl = count[count <= thrashold ].index

In [123]:
brand_ohe = pd.get_dummies(cars_df['brand'].replace(repl, 'uncommon'), dtype=int)

 THis is the reason we need a column transformer, to encode,scale etc of each column differently in a single pipeline.

 - See u in Day8

In [127]:
np.hstack((brand_ohe.values, x_train_new))

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 8128 and the array at index 1 has size 6502